# TADA Quickstart — Steering Audio Diffusion Models

> **TADA**: Tuning Audio Diffusion Models through Activation Steering (arXiv 2602.11910)

This notebook demonstrates the full steering pipeline in **dry-run mode** — no model
weights, GPU, or pre-computed steering vectors required.  Every step that would
normally call ACE-Step is replaced with synthetic tensors so you can explore the
API on any CPU machine.

---

## Contents

1. [Setup & imports](#1-setup)
2. [Create a SteeringVector](#2-steering-vector)
3. [SteeringVectorBank — save, load, summarise](#3-vector-bank)
4. [Multi-concept steering with Gram-Schmidt](#4-multi-concept)
5. [Timestep schedules](#5-timestep-schedules)
6. [Concept algebra — arithmetic on vectors](#6-concept-algebra)
7. [Unified SteeringPipeline](#7-pipeline)
8. [Evaluation metrics & alpha sweep](#8-eval-metrics)
9. [HF Hub upload / download (dry-run)](#9-hub)
10. [CLI commands (dry-run)](#10-cli)

---
## 1. Setup <a id='1-setup'></a>

In [ ]:
# Install the package in editable mode if running from a checkout.
# On Colab / a fresh environment run: pip install -e path/to/steer-audio
import sys, pathlib
REPO_ROOT = pathlib.Path().resolve().parent  # notebooks/ → repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import torch
import numpy as np

# Core steer_audio imports
from steer_audio.vector_bank import SteeringVector, SteeringVectorBank
from steer_audio.multi_steer import MultiConceptSteerer
from steer_audio.temporal_steering import (
    TimestepAdaptiveSteerer,
    constant_schedule,
    cosine_schedule,
    early_only_schedule,
    late_only_schedule,
)
from steer_audio.concept_algebra import ConceptAlgebra, ConceptFeatureSet
from steer_audio.pipeline import SteeringPipeline

print('steer_audio imported successfully')
print(f'PyTorch {torch.__version__}  |  device: {"cuda" if torch.cuda.is_available() else "cpu"}')

---
## 2. Create a SteeringVector <a id='2-steering-vector'></a>

A `SteeringVector` is a provenance-rich container for one CAA or SAE steering direction.
It stores the tensor, the concept name, which transformer layers to apply it to,
and quality metrics (`clap_delta`, `lpaps_at_50`).

In [ ]:
D_MODEL = 1024  # ACE-Step hidden dimension

# Synthetic unit-norm vector (stand-in for a real CAA vector)
v_tempo = torch.randn(D_MODEL)
v_tempo = v_tempo / v_tempo.norm()

sv_tempo = SteeringVector(
    concept="tempo",
    method="caa",
    model_name="ace-step",
    layers=[6, 7],          # functional layers identified by activation patching
    vector=v_tempo,
    clap_delta=0.12,        # ΔAlignment CLAP at alpha=50
    lpaps_at_50=0.08,       # LPAPS at alpha=50 (lower = better preservation)
    n_prompt_pairs=20,
)
print(sv_tempo)

In [ ]:
# Inspect key attributes
print(f'concept:   {sv_tempo.concept}')
print(f'method:    {sv_tempo.method}')
print(f'layers:    {sv_tempo.layers}')
print(f'dim:       {sv_tempo.vector.shape[0]}')
print(f'norm:      {sv_tempo.norm():.4f}')
print(f'clap_delta:{sv_tempo.clap_delta}')

### Save and reload

`SteeringVector.save()` writes a `.safetensors` file plus a human-readable `.json` sidecar.

In [ ]:
import tempfile, pathlib

with tempfile.TemporaryDirectory() as tmpdir:
    save_path = pathlib.Path(tmpdir) / "sv_tempo"
    sv_tempo.save(str(save_path))

    files = list(pathlib.Path(tmpdir).glob('*'))
    print('Saved files:', [f.name for f in files])

    sv_loaded = SteeringVector.load(str(save_path.with_suffix('.safetensors')))
    print('Loaded:', sv_loaded)
    print('Cosine similarity (self):', sv_tempo.cosine_similarity(sv_loaded))

---
## 3. SteeringVectorBank <a id='3-vector-bank'></a>

`SteeringVectorBank` is a persistent registry for multiple concepts.
It supports save/load, cosine composition, and a summary table.

In [ ]:
# Create a second concept vector
v_mood = torch.randn(D_MODEL)
v_mood = v_mood / v_mood.norm()

sv_mood = SteeringVector(
    concept="mood",
    method="caa",
    model_name="ace-step",
    layers=[6, 7],
    vector=v_mood,
    clap_delta=0.09,
    lpaps_at_50=0.11,
    n_prompt_pairs=20,
)

bank = SteeringVectorBank()
bank.add(sv_tempo)
bank.add(sv_mood)
print(f'Bank has {len(bank)} vectors')

In [ ]:
# Summary table
bank.summarise()

In [ ]:
# Cosine similarity between the two concept vectors
cos = sv_tempo.cosine_similarity(sv_mood)
print(f'cosine(tempo, mood) = {cos:.4f}  (near 0 → orthogonal directions)')

---
## 4. Multi-Concept Steering with Gram-Schmidt <a id='4-multi-concept'></a>

`MultiConceptSteerer` injects multiple steering vectors simultaneously.
With `orthogonalize=True` it applies Gram-Schmidt to remove cross-concept interference.

In [ ]:
steerer = MultiConceptSteerer(
    vectors={"tempo": sv_tempo, "mood": sv_mood},
    orthogonalize=True,
)
print(steerer)

# Compute the interference matrix (cosine similarities between vectors)
imat = steerer.interference_matrix(layer=6)
print(f'\nInterference matrix shape: {imat.shape}')
print('Concepts:', steerer.concepts)
print('Matrix:\n', imat.round(4))

In [ ]:
# Apply steering to a synthetic activation batch
# Shape: (batch=2, seq_len=16, d_model=1024)
h = torch.randn(2, 16, D_MODEL)

# Each concept gets its own alpha; steerer handles orthogonalization + ReNorm
h_steered = steerer.steer(h, alphas={"tempo": 50.0, "mood": 30.0}, layer=6)
print(f'Input shape:   {h.shape}')
print(f'Steered shape: {h_steered.shape}')
print(f'Max delta:     {(h_steered - h).abs().max().item():.4f}')

---
## 5. Timestep Schedules <a id='5-timestep-schedules'></a>

TADA supports per-step alpha schedules so the steering strength varies across
the denoising trajectory.  Four built-in schedules are available.

In [ ]:
T = 30   # total diffusion steps
alpha_max = 80.0

schedules = {
    "constant":   constant_schedule(alpha_max),
    "cosine":     cosine_schedule(alpha_max),
    "early_only": early_only_schedule(alpha_max),
    "late_only":  late_only_schedule(alpha_max),
}

for name, sched in schedules.items():
    values = [sched(t, T) for t in range(T)]
    print(f'{name:<12}: min={min(values):.1f}  max={max(values):.1f}  mean={np.mean(values):.1f}')

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 3))
    ts = list(range(T))
    for name, sched in schedules.items():
        ax.plot(ts, [sched(t, T) for t in ts], label=name)
    ax.set_xlabel('Diffusion step t')
    ax.set_ylabel('Effective alpha')
    ax.set_title('Timestep schedules (alpha_max=80)')
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed — skipping plot')

---
## 6. Concept Algebra <a id='6-concept-algebra'></a>

`ConceptAlgebra` lets you compose steering vectors with arithmetic expressions
like `jazz + female_vocal` or `fast_tempo - drums`.

In [ ]:
from steer_audio.concept_algebra import ConceptAlgebra, ConceptFeatureSet

# Build synthetic feature sets (in real use these come from SAE decomposition)
rng = np.random.default_rng(42)
n_features = 4096

def _random_feature_set(concept: str, k: int = 64) -> ConceptFeatureSet:
    """Synthetic top-k feature set."""
    indices = rng.choice(n_features, size=k, replace=False).tolist()
    weights = rng.random(k).tolist()
    return ConceptFeatureSet(
        concept=concept,
        feature_indices=indices,
        feature_weights=weights,
        model_name="ace-step",
        layer=7,
    )

features = {c: _random_feature_set(c) for c in ["jazz", "female_vocal", "fast_tempo", "drums"]}
algebra = ConceptAlgebra(features=features, d_model=D_MODEL)
print(algebra)

In [ ]:
# Arithmetic operations
result_add  = algebra.add("jazz", "female_vocal")
result_sub  = algebra.subtract("fast_tempo", "drums")
result_blend= algebra.blend([("jazz", 0.5), ("female_vocal", 0.5)])

print('jazz + female_vocal    :', result_add)
print('fast_tempo - drums     :', result_sub)
print('0.5*jazz + 0.5*female  :', result_blend)

In [ ]:
# Convert an algebra result to a SteeringVector ready for inference
sv_composed = algebra.to_steering_vector(result_add)
print(sv_composed)

---
## 7. Unified SteeringPipeline <a id='7-pipeline'></a>

`SteeringPipeline` is the unified Phase 2 entry point that composes all components:
vector bank, multi-concept steerer, timestep schedules, and concept algebra.

In [ ]:
from steer_audio.pipeline import SteeringPipeline
from steer_audio.vector_bank import SteeringVectorBank

bank2 = SteeringVectorBank()
bank2.add(sv_tempo)
bank2.add(sv_mood)

# Fluent API: chain add_concept() calls
pipeline = (
    SteeringPipeline(bank2, schedule_type="cosine")
    .add_concept("tempo", alpha=60)
    .add_concept("mood",  alpha=40)
)
print(pipeline)

In [ ]:
# dry_run=True returns a summary dict without calling any model
result = pipeline.generate("a jazz piano trio, upbeat", dry_run=True)
print('dry-run result keys:', list(result.keys()))
print('concepts:', result.get('concepts'))
print('alphas:  ', result.get('alphas'))
print('schedule:', result.get('schedule_type'))

---
## 8. Evaluation Metrics & Alpha Sweep <a id='8-eval-metrics'></a>

`EvalSuite` wraps CLAP, FAD, and LPAPS backends.  `StubBackend` returns fixed values
for testing; real backends require model weights.

In [ ]:
from steer_audio.eval_metrics import EvalSuite, MetricResult

# Use stub=True for dry-run (no model weights needed)
suite = EvalSuite(backends=["clap", "fad"], stub=True)
print('Available backends:', suite.availability())

In [ ]:
import tempfile, wave, struct, pathlib

def _write_dummy_wav(path: pathlib.Path) -> None:
    """Write a minimal valid WAV file (1 silent sample)."""
    with wave.open(str(path), 'w') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(44100)
        wf.writeframes(struct.pack('<h', 0))

with tempfile.TemporaryDirectory() as tmpdir:
    audio_dir = pathlib.Path(tmpdir)
    for i in range(3):
        _write_dummy_wav(audio_dir / f'sample_{i}.wav')

    result = suite.evaluate_dir(audio_dir, prompt="an upbeat jazz track")
    print('MetricResult:')
    for k, v in result.to_dict().items():
        print(f'  {k}: {v}')

---
## 9. HF Hub Upload / Download (dry-run) <a id='9-hub'></a>

`steer_audio.hub` provides `upload_vectors` and `download_vectors`.
Set `dry_run=True` to preview without touching the Hub.

In [ ]:
import logging, tempfile, pathlib
logging.basicConfig(level=logging.INFO)

from steer_audio.hub import upload_vectors, download_vectors

In [ ]:
# Upload dry-run: saves vectors to a temp dir first, then previews the upload
with tempfile.TemporaryDirectory() as tmpdir:
    local_dir = pathlib.Path(tmpdir)
    sv_tempo.save(str(local_dir / "sv_tempo"))
    sv_mood.save(str(local_dir / "sv_mood"))

    print('Files to upload:', [f.name for f in local_dir.glob('*')])

    url = upload_vectors(
        local_dir=local_dir,
        repo_id="myuser/tada-vectors",   # replace with your repo
        path_in_repo="vectors/ace-step",
        dry_run=True,                     # <-- no actual upload
    )
    print('\nDry-run URL:', url)

In [ ]:
# Download dry-run: preview which files would be fetched
# (requires huggingface_hub; skipped gracefully if not installed)
try:
    with tempfile.TemporaryDirectory() as tmpdir:
        files = download_vectors(
            repo_id="myuser/tada-vectors",
            local_dir=tmpdir,
            path_in_repo="vectors/ace-step",
            dry_run=True,
        )
        print('Would download:', files)
except ImportError as e:
    print(f'Skipped (huggingface_hub not installed): {e}')

---
## 10. CLI Commands (dry-run) <a id='10-cli'></a>

The `tada` CLI exposes every pipeline step as a subcommand.

In [ ]:
import subprocess, sys

def run(cmd: list[str]) -> None:
    result = subprocess.run(
        [sys.executable, '-m', 'steer_audio.cli'] + cmd,
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)

# List all available commands
run(['--help'])

In [ ]:
import os, tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    env_patch = {**os.environ, 'TADA_WORKDIR': tmpdir}

    cmds = [
        ['localize',         '--concept', 'tempo', '--dry-run'],
        ['compute-vectors',  '--concept', 'tempo', '--dry-run'],
        ['generate',         '--concept', 'tempo', '--alpha', '50', '--dry-run'],
        ['evaluate',         '--concept', 'tempo', '--dry-run'],
        ['train-sae',        '--layer', '7',       '--dry-run'],
        ['upload-vectors',   '--repo-id', 'myuser/tada-vectors', '--concept', 'tempo', '--dry-run'],
        ['download-vectors', '--repo-id', 'myuser/tada-vectors', '--concept', 'tempo', '--dry-run'],
    ]

    for cmd in cmds:
        result = subprocess.run(
            [sys.executable, '-m', 'steer_audio.cli'] + cmd,
            capture_output=True, text=True, env=env_patch,
        )
        print(f'tada {" ".join(cmd[:1])}: {(result.stdout or result.stderr).strip()[:80]}')

---
## Next Steps

| Task | Command |
|------|---------|
| Generate real ACE-Step activations | `python sae/sae_src/sae/cache_activations_runner_ace.py` |
| Compute CAA steering vectors | `tada compute-vectors --concept tempo` |
| Run alpha sweep evaluation | `tada evaluate --concept tempo` |
| Train SAE on layer 7 activations | `tada train-sae --layer 7` |
| Upload vectors to HF Hub | `HF_TOKEN=... tada upload-vectors --repo-id myuser/tada-vectors` |
| Explore the Gradio UI | `python demo/app.py` |

See [`docs/results_summary.md`](../docs/results_summary.md) for a summary of all
experiment results, and [`docs/scaling_real_runs.md`](../docs/scaling_real_runs.md)
for instructions on running real ACE-Step experiments.